# Data exploration and cleaning study

This notebook studies the supplied employees, projects and timesheets before implementing a modular ETL pipeline. It preserves the source data, profiles quality issues, defines explicit cleaning decisions, and builds **in-memory candidate datasets** with an auditable issue table. It does not overwrite CSVs or load a database.

Run all cells from top to bottom with Python 3 and pandas, with the working directory set to the folder containing this notebook and the three CSVs. If needed, install pandas in your notebook environment (`%pip install pandas`). All results are derived from the files, not from previous notebook state.

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path.cwd()
COLUMNS = {
    "employees": ["employee_id", "name", "role"],
    "projects": ["project_id", "project_name", "budget"],
    "timesheets": ["employee_id", "project_id", "date", "hours"],
}
raw = {}
for table, columns in COLUMNS.items():
    # Preserve original text, including empty fields; do not infer IDs/numbers/dates.
    frame = pd.read_csv(DATA_DIR / f"{table}.csv", dtype="string", keep_default_na=False)
    assert frame.columns.tolist() == columns, f"Unexpected schema: {table}"
    frame.insert(0, "source_record", range(1, len(frame) + 1))
    raw[table] = frame

# source_record is the 1-based parsed record ordinal, not a physical file line.
display(pd.DataFrame([
    {"table": t, "rows": len(df), "business_columns": len(COLUMNS[t])}
    for t, df in raw.items()
]))
for table, frame in raw.items():
    print(table)
    display(frame.head())

,table,rows,business_columns
0,employees,42,3
1,projects,39,3
2,timesheets,343,4


employees


,source_record,employee_id,name,role
0,1,E001,Sarah Okonkwo,Data Engineer
1,2,E002,James Patel,Data Analyst
2,3,E003,Maria Gonzalez,Backend Engineer
3,4,E004,Tom Whitfield,Product Manager
4,5,E005,Lena Bauer,Data Scientist


projects


,source_record,project_id,project_name,budget
0,1,P001,Alpha Platform Rebuild,120000
1,2,P002,Data Warehouse Migration,85000
2,3,P003,Customer Portal v2,60000
3,4,P004,Internal Analytics Dashboard,45000
4,5,P005,Mobile App Launch,200000


timesheets


,source_record,employee_id,project_id,date,hours
0,1,E001,P001,07/01/2024,8
1,2,E001,P001,08/01/2024,7.5
2,3,E002,P003,07/01/2024,6
3,4,E003,P002,07/01/2024,8
4,5,E004,P004,2024-01-08,4


## 1. Profile completeness, types and duplicates

Read everything as text first: numeric inference could hide bad values and automatic missing-value inference could erase literal strings. Strip surrounding whitespace and represent empty strings as `pd.NA` in a separate working copy. Names keep their accents, spelling and case. IDs are not silently repaired or uppercased.

Duplicate comparisons exclude the provenance column. Separate repeated identical records from **conflicting records sharing an identifier**; keeping the last row is not justified without a version or update timestamp.

In [2]:
prepared = {}
profile = []
for table, source in raw.items():
    frame = source.copy(deep=True)
    for column in COLUMNS[table]:
        trimmed = frame[column].str.strip()
        frame[column] = trimmed.mask(trimmed.eq(""), pd.NA)
        profile.append({
            "table": table, "column": column, "loaded_dtype": str(source[column].dtype),
            "missing_after_trim": int(frame[column].isna().sum()),
            "whitespace_changes": int(source[column].ne(trimmed).sum()),
            "distinct_nonmissing": frame[column].nunique(),
        })
    prepared[table] = frame

display(pd.DataFrame(profile))
for table, frame in prepared.items():
    repeated = frame.duplicated(COLUMNS[table], keep=False)
    print(f"{table}: {frame.duplicated(COLUMNS[table]).sum()} redundant copies")
    display(frame.loc[repeated])

for table, key in [("employees", "employee_id"), ("projects", "project_id")]:
    distinct = prepared[table].drop_duplicates(COLUMNS[table])
    conflicts = distinct[key].notna() & distinct.duplicated(key, keep=False)
    print(f"{table}: conflicting identifiers")
    display(distinct.loc[conflicts])

print("Role vocabulary (no approved role list was supplied)")
display(prepared["employees"]["role"].value_counts(dropna=False).rename("rows").to_frame())

,table,column,loaded_dtype,missing_after_trim,whitespace_changes,distinct_nonmissing
0,employees,employee_id,string,0,0,40
1,employees,name,string,1,0,39
2,employees,role,string,1,0,8
3,projects,project_id,string,1,0,35
4,projects,project_name,string,1,0,35
5,projects,budget,string,1,0,35
6,timesheets,employee_id,string,1,0,42
7,timesheets,project_id,string,0,0,22
8,timesheets,date,string,0,0,106
9,timesheets,hours,string,1,0,18


employees: 2 redundant copies


,source_record,employee_id,name,role
0,1,E001,Sarah Okonkwo,Data Engineer
10,11,E001,Sarah Okonkwo,Data Engineer
14,15,E014,Anouk Vermeer,Data Scientist
31,32,E014,Anouk Vermeer,Data Scientist


projects: 3 redundant copies


,source_record,project_id,project_name,budget
0,1,P001,Alpha Platform Rebuild,120000
6,7,P007,ML Forecasting Pipeline,95000
10,11,P001,Alpha Platform Rebuild,120000
20,21,P007,ML Forecasting Pipeline,95000
28,29,P026,Experimentation Platform,67000
33,34,P026,Experimentation Platform,67000


...

,source_record,employee_id,name,role


projects: conflicting identifiers


,source_record,project_id,project_name,budget


Role vocabulary (no approved role list was supplied)


,rows
role,
Data Engineer,9
Data Analyst,8
Backend Engineer,6
Data Scientist,6
Product Manager,4
Analytics Engineer,4
DevOps Engineer,3
Project Manager,1
<NA>,1


## 2. Study numeric values and date formats

Use numeric conversion only to diagnose failures, never to replace missing or invalid values with zero. The project budget `55000-60000` is a range, not a scalar; choosing its midpoint would invent a business rule. Likewise, `seven` is left for review rather than automatically translated.

Dates contain slash dates, ISO dates and a textual month. **Assumption:** slash dates are day/month/year, supported by dates such as `14/01/2024`; this still needs source-owner confirmation. Match explicit formats rather than asking pandas to guess. The textual-month parser below maps English months explicitly so it does not depend on the machine locale.

In [3]:
def parse_dates(values):
    parsed = pd.Series(pd.NaT, index=values.index, dtype="datetime64[ns]")
    formats = pd.Series("unsupported", index=values.index, dtype="string")
    patterns = [
        (r"\d{2}/\d{2}/\d{4}", "%d/%m/%Y", "day/month/year"),
        (r"\d{4}-\d{2}-\d{2}", "%Y-%m-%d", "ISO"),
    ]
    for pattern, fmt, label in patterns:
        mask = values.str.fullmatch(pattern, na=False)
        parsed.loc[mask] = pd.to_datetime(values.loc[mask], format=fmt, errors="coerce")
        formats.loc[mask] = label
    months = {name: f"{n:02}" for n, name in enumerate(
        ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], 1)}
    parts = values.str.extract(r"^(\d{2})-([A-Z][a-z]{2})-(\d{2})$")
    mask = parts[1].isin(months)
    # Explicit study assumption: two-digit years denote 2000–2099.
    iso = "20" + parts.loc[mask, 2] + "-" + parts.loc[mask, 1].map(months) + "-" + parts.loc[mask, 0]
    parsed.loc[mask] = pd.to_datetime(iso, format="%Y-%m-%d", errors="coerce")
    formats.loc[mask] = "English month / two-digit year"
    formats.loc[values.isna()] = "missing"
    return parsed, formats

study = {table: frame.copy(deep=True) for table, frame in prepared.items()}
study["projects"]["budget_numeric"] = pd.to_numeric(study["projects"]["budget"], errors="coerce")
study["timesheets"]["hours_numeric"] = pd.to_numeric(study["timesheets"]["hours"], errors="coerce")
study["timesheets"]["work_date"], date_formats = parse_dates(study["timesheets"]["date"])
display(date_formats.value_counts(dropna=False).rename("rows").to_frame())
for table, column in [("projects", "budget"), ("timesheets", "hours")]:
    frame = study[table]
    numeric = frame[f"{column}_numeric"]
    print(f"{table}: numeric distribution (before validation)")
    display(numeric.describe().to_frame())
    bad = numeric.isna() | numeric.isin([float("inf"), -float("inf")]) | numeric.lt(0)
    if column == "hours":
        bad |= numeric.eq(0) | numeric.gt(24)
    display(frame.loc[bad.fillna(True)])

print("Unparseable dates")
display(study["timesheets"].loc[study["timesheets"]["work_date"].isna()])
print("Observed date range:", study["timesheets"]["work_date"].min(), "to", study["timesheets"]["work_date"].max())

,rows
day/month/year,341
ISO,1
English month / two-digit year,1


projects: numeric distribution (before validation)


,budget_numeric
count,37.0
mean,67270.27027
std,40065.674314
min,-5000.0
25%,40000.0
50%,62000.0
75%,91000.0
max,200000.0


,source_record,project_id,project_name,budget,budget_numeric
7,8,P008,API Gateway Consolidation,55000-60000,<NA>
21,22,P019,Data Quality Framework,-5000,-5000
35,36,P032,Supply Chain Analytics,<NA>,<NA>


timesheets: numeric distribution (before validation)


,hours_numeric
count,340.0
mean,7.232353
std,1.670624
min,-3.0
25%,6.5
50%,7.5
75%,8.0
max,25.0


,source_record,employee_id,project_id,date,hours,hours_numeric,work_date
61,62,E014,P007,29/01/2024,-3,-3.0,2024-01-29
67,68,E020,P003,30/01/2024,25,25.0,2024-01-30
202,203,E005,P007,31/03/2024,<NA>,<NA>,2024-03-31
272,273,E020,P003,01/05/2024,not_a_number,<NA>,2024-05-01
330,331,E030,P009,27/05/2024,seven,<NA>,2024-05-27


Unparseable dates


,source_record,employee_id,project_id,date,hours,hours_numeric,work_date


Observed date range: 2024-01-07 00:00:00 to 2024-05-30 00:00:00


## 3. Study references and the timesheet grain

Expected entities are employees and projects, connected by timesheets. Missing references must be checked against the original known identifiers **and** the eventual accepted dimensions: a known project can still be held for review.

There is no timesheet entry ID. For this study, assume one record per employee, project and day. Identical repeated entries are treated as duplicate ingestion; conflicting hours at the same grain require review. Confirm this assumption before a production uniqueness constraint: multiple legitimate entries per day would require a source entry ID instead.

In [4]:
times = study["timesheets"]
for column, parent in [("employee_id", "employees"), ("project_id", "projects")]:
    known = prepared[parent][column].dropna()
    missing_reference = times[column].notna() & ~times[column].isin(known)
    print(f"Unknown {column} values")
    display(times.loc[missing_reference])

GRAIN = ["employee_id", "project_id", "work_date"]
semantic_columns = GRAIN + ["hours_numeric"]
parseable = times.loc[times[semantic_columns].notna().all(axis=1)]
print("Repeated employee/project/day groups (includes identical copies)")
display(parseable.loc[parseable.duplicated(GRAIN, keep=False)].sort_values(GRAIN))
print("Daily hours before final validation, excluding identical parsed copies")
daily_study = (parseable.drop_duplicates(semantic_columns)
               .groupby(["employee_id", "work_date"], as_index=False)["hours_numeric"].sum())
display(daily_study.loc[daily_study["hours_numeric"].gt(12)])

Unknown employee_id values


,source_record,employee_id,project_id,date,hours,hours_numeric,work_date
56,57,E099,P001,25/01/2024,8,8.0,2024-01-25
200,201,E050,P001,31/03/2024,8,8.0,2024-03-31


Unknown project_id values


,source_record,employee_id,project_id,date,hours,hours_numeric,work_date
57,58,E010,P999,28/01/2024,7.5,7.5,2024-01-28
201,202,E001,P999,31/03/2024,6,6.0,2024-03-31


Repeated employee/project/day groups (includes identical copies)


,source_record,employee_id,project_id,date,hours,hours_numeric,work_date
156,157,E001,P001,11/03/2024,7.5,7.5,2024-03-11
159,160,E001,P001,11/03/2024,7.5,7.5,2024-03-11
5,6,E005,P007,08/01/2024,9,9.0,2024-01-08
6,7,E005,P007,08/01/2024,9,9.0,2024-01-08


Daily hours before final validation, excluding identical parsed copies


,employee_id,work_date,hours_numeric
183,E020,2024-01-30,25.0


## 4. Proposed cleaning and disposition rules

These are explicit study policies, not confirmed business requirements. **Rejected** means unusable as supplied or a redundant copy; **review** means excluded from the candidate clean set until resolved; **accepted** means no blocking issues. Multiple issues can belong to one row. Rejection takes precedence over review.

| Check | Decision |
|---|---|
| Surrounding whitespace / empty fields | Trim / standardize missing values on copies; preserve raw inputs. |
| Missing or malformed IDs | Reject. Provisional ID pattern: `E` or `P` plus three digits. |
| Identical dimension rows | Keep first in file order; reject subsequent copies with a reason. |
| Conflicting dimension ID | Review all distinct versions; do not pick a winner. |
| Missing employee name/role or project name | Review; do not fabricate a label. |
| Missing, nonnumeric, infinite or negative budget | Review. Zero is permitted. Do not guess a range's value or currency. |
| Missing, nonnumeric, infinite, zero, negative or >24 hours | Reject. Assume positive work entries; adjustment/zero-entry policies need confirmation. |
| Missing, unsupported or impossible date | Reject; parse only documented formats. No runtime-dependent “future date” rule. |
| Unknown reference | Reject. A syntactically valid ID is not necessarily a known entity. |
| Reference to an unaccepted dimension | Review; exclude from strict analysis-ready candidates. |
| Identical parsed timesheet | Keep first; reject extra copies under the stated grain assumption. |
| Conflicting employee/project/day entries | Review all versions; do not sum or choose arbitrarily. |
| Employee/day total >24, after basic validation and deduplication | Reject every contributing row: impossible under the positive-hours assumption. |
| Employee/day total >12 and ≤24 | Review every contributing row; 12 is a provisional review threshold, not a legal limit. |

Weekend work and uncommon roles are not automatically errors. Missing descriptive dimension fields need not invalidate the actual work; this study deliberately uses strict dimension acceptance, with dependent timesheets held for review. An alternative later model could allow nullable descriptions and quality flags.

In [5]:
# Small notebook helpers keep the study readable; this is not yet the ETL package.
issues = []
def flag(table, mask, column, rule, disposition, explanation):
    frame = study[table]
    for index in frame.index[mask.fillna(False)]:
        issues.append({
            "table": table, "source_record": int(frame.at[index, "source_record"]),
            "column": column, "raw_value": raw[table].at[index, column] if column in raw[table] else "",
            "rule": rule, "disposition": disposition, "explanation": explanation,
        })

def statuses(table):
    result = pd.Series("accepted", index=study[table].index, dtype="string")
    for disposition in ["review", "rejected"]:
        records = {i["source_record"] for i in issues if i["table"] == table and i["disposition"] == disposition}
        result.loc[study[table]["source_record"].isin(records)] = disposition
    return result

for table, frame in study.items():
    for column in [c for c in COLUMNS[table] if c.endswith("_id")]:
        prefix = "E" if column == "employee_id" else "P"
        flag(table, frame[column].isna(), column, "missing_id", "rejected", "Identifier is required.")
        flag(table, frame[column].notna() & ~frame[column].str.fullmatch(prefix + r"\d{3}", na=False),
             column, "invalid_id_format", "rejected", "Expected prefix plus three digits.")

for table, key, descriptions in [
    ("employees", "employee_id", ["name", "role"]),
    ("projects", "project_id", ["project_name"]),
]:
    frame = study[table]
    copies = frame.duplicated(COLUMNS[table], keep="first")
    flag(table, copies, key, "duplicate_record", "rejected", "Redundant identical row; retain first source record.")
    distinct = frame.loc[~copies]
    conflicting_ids = distinct.loc[distinct[key].notna() & distinct.duplicated(key, keep=False), key]
    flag(table, frame[key].isin(conflicting_ids), key, "conflicting_id", "review", "Different records share one ID.")
    for column in descriptions:
        flag(table, frame[column].isna(), column, "missing_description", "review", "Request missing descriptive value.")

budget = study["projects"]["budget_numeric"]
flag("projects", budget.isna() | budget.isin([float("inf"), -float("inf")]) | budget.lt(0),
     "budget", "invalid_or_missing_budget", "review", "Require a finite nonnegative scalar budget; do not impute.")

t = study["timesheets"]
hours = t["hours_numeric"]
flag("timesheets", hours.isna(), "hours", "missing_or_nonnumeric_hours", "rejected", "Require numeric hours; no imputation.")
flag("timesheets", hours.notna() & ~hours.between(0, 24, inclusive="right"),
     "hours", "hours_out_of_range", "rejected", "Require 0 < hours <= 24.")
flag("timesheets", t["work_date"].isna(), "date", "invalid_or_missing_date", "rejected", "Date must parse using an approved format.")

for column, parent in [("employee_id", "employees"), ("project_id", "projects")]:
    known = study[parent][column].dropna()
    accepted_ids = study[parent].loc[statuses(parent).eq("accepted"), column]
    flag("timesheets", t[column].notna() & ~t[column].isin(known), column,
         "unknown_reference", "rejected", "ID does not exist in source dimension.")
    flag("timesheets", t[column].isin(known) & ~t[column].isin(accepted_ids), column,
         "unaccepted_dimension", "review", "Known dimension record is not accepted; resolve upstream issue.")

# Semantic comparison catches equivalent date/number spellings as well as raw copies.
comparable = t[semantic_columns].notna().all(axis=1)
unique_times = t.loc[comparable]
duplicate_indices = unique_times.index[unique_times.duplicated(semantic_columns, keep="first")]
flag("timesheets", pd.Series(t.index.isin(duplicate_indices), index=t.index), "hours",
     "duplicate_record", "rejected", "Redundant parsed entry; retain first source record.")
distinct_times = unique_times.drop_duplicates(semantic_columns)
conflict_keys = distinct_times.loc[distinct_times.duplicated(GRAIN, keep=False), GRAIN]
conflict_mask = pd.MultiIndex.from_frame(t[GRAIN]).isin(pd.MultiIndex.from_frame(conflict_keys))
flag("timesheets", pd.Series(conflict_mask, index=t.index), "hours",
     "conflicting_timesheet_grain", "review", "Same employee/project/day has different hours.")

# Include review rows with valid work values; exclude hard failures and duplicate copies.
daily_candidates = t.loc[~statuses("timesheets").eq("rejected")]
daily_totals = daily_candidates.groupby(["employee_id", "work_date"])["hours_numeric"].transform("sum")
for mask, rule, disposition, explanation in [
    (daily_totals.gt(24), "daily_hours_over_24", "rejected", "Employee/day total exceeds 24; all contributing entries rejected."),
    (daily_totals.gt(12) & daily_totals.le(24), "long_work_day", "review", "Employee/day total exceeds provisional 12-hour review threshold."),
]:
    flag("timesheets", mask.reindex(t.index, fill_value=False), "hours", rule, disposition, explanation)

validation_issues = pd.DataFrame(issues, columns=[
    "table", "source_record", "column", "raw_value", "rule", "disposition", "explanation"
]).sort_values(["table", "source_record", "rule"]).reset_index(drop=True)
classified = {table: frame.assign(status=statuses(table)) for table, frame in study.items()}
print("Issue counts (one record may have several issues)")
display(validation_issues.groupby(["table", "rule", "disposition"]).size().rename("issues").reset_index())
print("Full row-level audit")
with pd.option_context("display.max_rows", None, "display.max_colwidth", 100):
    display(validation_issues)

Issue counts (one record may have several issues)


,table,rule,disposition,issues
0,employees,duplicate_record,rejected,2
1,employees,missing_description,review,2
2,projects,duplicate_record,rejected,3
3,projects,invalid_or_missing_budget,review,3
4,projects,missing_description,review,1
5,projects,missing_id,rejected,1
6,timesheets,duplicate_record,rejected,2
7,timesheets,hours_out_of_range,rejected,2
8,timesheets,missing_id,rejected,1
9,timesheets,missing_or_nonnumeric_hours,rejected,3


Full row-level audit


,table,source_record,column,raw_value,rule,disposition,explanation
0,employees,11,employee_id,E001,duplicate_record,rejected,Redundant identical row; retain first source record.
1,employees,14,name,,missing_description,review,Request missing descriptive value.
2,employees,22,role,,missing_description,review,Request missing descriptive value.
3,employees,32,employee_id,E014,duplicate_record,rejected,Redundant identical row; retain first source record.
4,projects,8,budget,55000-60000,invalid_or_missing_budget,review,Require a finite nonnegative scalar budget; do not impute.
5,projects,11,project_id,P001,duplicate_record,rejected,Redundant identical row; retain first source record.
6,projects,14,project_name,,missing_description,review,Request missing descriptive value.
7,projects,15,project_id,,missing_id,rejected,Identifier is required.
8,projects,21,project_id,P007,duplicate_record,rejected,Redundant identical row; retain first source record.
9,projects,22,budget,-5000,invalid_or_missing_budget,review,Require a finite nonnegative scalar budget; do not impute.


## 5. Candidate cleaning results and reconciliation

The following datasets are previews held in memory. Review and rejected records retain all original business fields, source ordinals and parsed helper values. `validation_issues` explains each decision. Clean candidates use typed dates/numbers and keep provenance; raw CSVs remain unchanged.

Reconciliation counts records, not issues. The assertions check referential integrity, uniqueness, ranges and full row accounting. Budget values here are exploratory pandas numbers; a later SQL schema should use fixed-precision `NUMERIC` for money.

In [6]:
accepted = {table: frame.loc[frame["status"].eq("accepted")].copy() for table, frame in classified.items()}
review = {table: frame.loc[frame["status"].eq("review")].copy() for table, frame in classified.items()}
rejected = {table: frame.loc[frame["status"].eq("rejected")].copy() for table, frame in classified.items()}
clean_employees = accepted["employees"][["source_record", "employee_id", "name", "role"]].copy()
clean_projects = accepted["projects"][["source_record", "project_id", "project_name", "budget_numeric"]].rename(columns={"budget_numeric": "budget"})
clean_timesheets = accepted["timesheets"][["source_record", "employee_id", "project_id", "work_date", "hours_numeric"]].rename(columns={"work_date": "date", "hours_numeric": "hours"})

reconciliation = pd.DataFrame([
    {"table": table, "raw": len(raw[table]), "accepted": len(accepted[table]),
     "review": len(review[table]), "rejected": len(rejected[table])}
    for table in raw
]).set_index("table")
assert reconciliation[["accepted", "review", "rejected"]].sum(axis=1).eq(reconciliation["raw"]).all()
for table, frame in classified.items():
    assert frame["source_record"].is_unique
    assert len(frame) == len(raw[table])
assert clean_employees["employee_id"].is_unique
assert clean_projects["project_id"].is_unique
assert not clean_timesheets.duplicated(["employee_id", "project_id", "date"]).any()
assert clean_timesheets["employee_id"].isin(clean_employees["employee_id"]).all()
assert clean_timesheets["project_id"].isin(clean_projects["project_id"]).all()
assert clean_timesheets["hours"].between(0, 24, inclusive="right").all()
assert clean_timesheets["date"].notna().all()
assert clean_projects["budget"].ge(0).all()
assert clean_timesheets.groupby(["employee_id", "date"])["hours"].sum().le(12).all()
display(reconciliation)
print("Candidate timesheets:", len(clean_timesheets), "rows;", clean_timesheets["hours"].sum(), "hours")
display(clean_timesheets.head(10))
for table in COLUMNS:
    print(f"{table}: records requiring review")
    display(review[table])
print("All reconciliation and candidate integrity checks passed.")

,raw,accepted,review,rejected
table,,,,
employees,42,38,2,2
projects,39,31,4,4
timesheets,343,309,22,12


Candidate timesheets: 309 rows; 2220.5 hours


,source_record,employee_id,project_id,date,hours
0,1,E001,P001,2024-01-07,8.0
1,2,E001,P001,2024-01-08,7.5
2,3,E002,P003,2024-01-07,6.0
3,4,E003,P002,2024-01-07,8.0
4,5,E004,P004,2024-01-08,4.0
5,6,E005,P007,2024-01-08,9.0
7,8,E006,P006,2024-01-09,8.0
8,9,E007,P001,2024-01-09,7.0
9,10,E008,P010,2024-01-09,6.5
10,11,E009,P003,2024-01-10,8.0


employees: records requiring review


,source_record,employee_id,name,role,status
13,14,E013,<NA>,Data Analyst,review
21,22,E021,Leo Dubois,<NA>,review


projects: records requiring review


,source_record,project_id,project_name,budget,budget_numeric,status
7,8,P008,API Gateway Consolidation,55000-60000,<NA>,review
13,14,P013,<NA>,15000,15000,review
21,22,P019,Data Quality Framework,-5000,-5000,review
35,36,P032,Supply Chain Analytics,<NA>,<NA>,review


timesheets: records requiring review


,source_record,employee_id,project_id,date,hours,hours_numeric,work_date,status
14,15,E013,P009,11/01/2024,8,8.0,2024-01-11,review
16,17,E015,P008,11/01/2024,8,8.0,2024-01-11,review
24,25,E003,P008,15/01/2024,8,8.0,2024-01-15,review
31,32,E024,P008,17/01/2024,8,8.0,2024-01-17,review
60,61,E013,P009,28/01/2024,6,6.0,2024-01-28,review
62,63,E015,P008,29/01/2024,8,8.0,2024-01-29,review
68,69,E021,P004,30/01/2024,6,6.0,2024-01-30,review
71,72,E024,P008,31/01/2024,8,8.0,2024-01-31,review
182,183,E003,P008,21/03/2024,7.5,7.5,2024-03-21,review
183,184,E015,P008,21/03/2024,8,8.0,2024-03-21,review


All reconciliation and candidate integrity checks passed.


## 6. Findings and handoff to the later pipeline

The executed outputs above are the source of truth for counts. The supplied data contains:

- Identical employee and project duplicates, plus repeated timesheet entries. No arbitrary “keep last ID” rule is necessary.
- Missing employee names/roles and project names; one project has no ID.
- A project budget range, a negative budget and a missing budget.
- Unknown timesheet employees (`E099`, `E050`) and project (`P999`), a missing employee, negative and excessive hours, missing hours and nonnumeric hours (`not_a_number`, `seven`).
- Three date representations. Mixed formatting is repairable under the documented interpretation; it is distinct from an invalid calendar date.
- Validly referenced timesheets whose employee descriptions or project budget require review. Filtering only unknown IDs would miss these dependencies.

**Decisions to confirm:** date locale and two-digit-year interpretation, one-entry-per-employee/project/day grain, positive-only hours, the 12-hour review threshold, budget currency and treatment of incomplete dimension descriptions. No values are inferred from names, neighboring records, or current time.

**Next implementation step (outside this notebook's scope):** extract these agreed policies into separate raw ingestion, normalization, dimension validation, timesheet validation, and transformation modules. Preserve source provenance and the issue ledger; publish accepted, review and rejected records separately. Derive project/employee hour summaries only from accepted timesheets and validate joins as many-to-one. Design the schema and ER diagram after confirming the grain.

**Safe reruns:** this notebook reconstructs every intermediate object and clears the issue list when run top to bottom. It performs no writes to input files and no appends to output datasets. Source order deterministically selects the retained duplicate. For later persisted runs, use input fingerprints, a rule version, stable source entry IDs where available, and atomic replacement or transactional loads; a record ordinal alone is not a durable business ID.

**Database vs pipeline:** the eventual database should enforce primary/foreign keys, required typed values and row-level range checks. Parsing, raw-value preservation, conflict detection, review routing and employee/day aggregate checks belong in the pipeline (or need explicit database triggers/transaction logic). Do not impose a timesheet composite unique key until the grain is confirmed.